## Writing a video using OpenCV
While building applications, it becomes important to save demo videos of your work as well as many applications themselves might require saving a video clip. For example, in a surveiallance application, you might have to save a video clip as soon as you see something unusual happening.

In this notebook, we will describe how to save a video in **avi** and **mp4** formats using OpenCV.

In [ ]:
# Import video processing, plotting, and portable file/process helpers.
import os
import shutil
import subprocess
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

from zipfile import ZipFile
from urllib.request import urlretrieve

from IPython.display import YouTubeVideo, display, Video

%matplotlib inline

##  <font style="color:black">Download Assets</font>

In [ ]:
def download_and_unzip(url, save_path):
    print("Downloading and extracting assets...", end="")
    urlretrieve(url, save_path)

    # Let download/extraction errors stop execution rather than use missing assets.
    with ZipFile(save_path) as z:
        z.extractall(Path(save_path).parent)
    print("Done")

In [ ]:
URL = r"https://www.dropbox.com/s/p8h7ckeo2dn1jtz/opencv_bootcamp_assets_NB6.zip?dl=1"

# Support kernels launched from either the repository root or this notebook folder.
asset_dir = Path.cwd()
repo_asset_dir = asset_dir / "notebooks" / "06-video-writing"
if repo_asset_dir.is_dir():
    asset_dir = repo_asset_dir
asset_zip_path = asset_dir / "opencv_bootcamp_assets_NB6.zip"
source_path = asset_dir / "race_car.mp4"

# An existing ZIP may still need extracting if the source video is missing.
if not source_path.exists():
    if asset_zip_path.exists():
        with ZipFile(asset_zip_path) as z:
            z.extractall(asset_dir)
    else:
        download_and_unzip(URL, asset_zip_path)
if not source_path.exists():
    raise FileNotFoundError(f"Missing source video: {source_path}")

## Read Video from Source

In [ ]:
source = str(source_path)  # Use integer 0 for a webcam.
cap = cv2.VideoCapture(source)

In [ ]:
if not cap.isOpened():
    cap.release()
    raise RuntimeError(f"Could not open video source: {source}")

### <font color="green">Read and display one frame </font>

In [ ]:
# ret is True only when a frame was decoded successfully.
ret, frame = cap.read()
if not ret:
    cap.release()
    raise RuntimeError("The video source did not return a readable first frame.")

In [ ]:
# Reverse the BGR channel axis to RGB for Matplotlib; shape remains (H, W, 3).
plt.imshow(frame[..., ::-1])

### <font color="green">Display the video file </font>

In [ ]:
video = YouTubeVideo("RwxVEjv78LQ", width=700, height=438)
display(video)

## Write Video using OpenCV
For writing the video, you need to create a videowriter object with the right parameters.

### <font color="green">Function Syntax</font>

```python
VideoWriter object = cv.VideoWriter(filename, fourcc, fps, frameSize )
```
where,
**Parameters**

1. `filename`: Name of the output video file.

2. `fourcc`: 4-character code of codec used to compress the frames. For example, `VideoWriter::fourcc('P','I','M','1')` is a MPEG-1 codec, VideoWriter::fourcc('M','J','P','G') is a motion-jpeg codec etc. List of codes can be obtained at Video Codecs by FOURCC page. FFMPEG backend with MP4 container natively uses other values as fourcc code: see ObjectType, so you may receive a warning message from OpenCV about fourcc code conversion.

3. `fps`: Framerate of the created video stream.

4. `frameSize`: Size of the video frames.

In [ ]:
# Array shape is (height, width, channels); VideoWriter expects (width, height).
frame_height, frame_width = frame.shape[:2]
fps = cap.get(cv2.CAP_PROP_FPS)
# Some webcams/backends cannot report FPS; choose a usable rate in that case.
if not (0 < fps < float("inf")):
    fps = 30.0

avi_path = asset_dir / "race_car_out.avi"
mp4_path = asset_dir / "race_car_out.mp4"
# MJPG suits AVI; mp4v avoids requesting the AVI-oriented XVID tag in MP4.
out_avi = cv2.VideoWriter(str(avi_path), cv2.VideoWriter_fourcc(*"MJPG"), fps, (frame_width, frame_height))
out_mp4 = cv2.VideoWriter(str(mp4_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (frame_width, frame_height))
if not out_avi.isOpened() or not out_mp4.isOpened():
    cap.release()
    out_avi.release()
    out_mp4.release()
    raise RuntimeError("Could not open both video writers; check codec support and output paths.")

### <font color="green">Read frames and write to file</font>
We will read the frames from the race-car video and write the same to the two objects we created in the previous step. We should release the objects after the task is complete.

In [ ]:
# Include the first frame already read for the preview, then read the rest.
# Always release resources, even if writing raises or the cell is interrupted.
try:
    while ret:
        out_avi.write(frame)
        out_mp4.write(frame)
        ret, frame = cap.read()
finally:
    cap.release()
    out_avi.release()
    out_mp4.release()

In [ ]:
# The previous cell closed the writers, finalizing the video containers.
print(f"Saved {avi_path.name} and {mp4_path.name} at {fps:g} FPS.")

For notebook playback, convert the generated MP4 to H.264 with FFmpeg. Browser support depends on the video codec as well as the container; OpenCV's available encoders depend on its build and backend.

The next cell installs `imageio-ffmpeg`, which supplies an FFmpeg executable on supported Windows, macOS, and Linux systems. If FFmpeg is already on PATH, the conversion uses it.

In [ ]:
# Install into the current notebook kernel's environment (also works on Windows).
%pip install imageio-ffmpeg

Convert only the MP4 output for browser playback. The original AVI and MP4 files remain available as OpenCV writing examples.

In [ ]:
# Prefer a system FFmpeg; otherwise use the executable from imageio-ffmpeg.
ffmpeg_exe = shutil.which("ffmpeg")
if ffmpeg_exe is None:
    import imageio_ffmpeg
    ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()

playback_path = asset_dir / "race_car_out_x264.mp4"
# A list handles paths with spaces; check=True stops on conversion failure.
# yuv420p improves browser compatibility; faststart moves metadata before frames.
subprocess.run(
    [ffmpeg_exe, "-y", "-i", str(mp4_path), "-c:v", "libx264",
     "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(playback_path),
     "-hide_banner", "-loglevel", "error"],
    check=True,
)

Render MP4 Video

In [ ]:
# Embed the local file so playback does not depend on a Colab /content path.
Video(str(playback_path), embed=True, width=700)

The video rendered in the above cell should be the same as the following.

In [ ]:
video = YouTubeVideo("2Gju7YLfkP0", width=700, height=438)
display(video)

## Thank You!